# Reduced-Dropout Neural Network Experiment

## Goal

Test whether the baseline neural network is slightly over-regularized.

The baseline MLP performed close to Histogram Gradient Boosting but remained
approximately 1.24% worse in validation RMSE.

This experiment keeps the dataset, chronological split, preprocessing, hidden-layer
sizes, optimizer, learning rate, weight decay, scheduler, batch size, and early
stopping procedure unchanged.

Only the dropout probabilities are reduced.

## Hypothesis

The baseline model may benefit from slightly weaker dropout regularization.
Reducing dropout may allow the network to learn more detailed relationships while
still retaining enough regularization to generalize well.

## Controlled Change

Baseline:

- First dropout: 0.20
- Second dropout: 0.10

Experiment:

- First dropout: 0.10
- Second dropout: 0.05

In [38]:
import copy
import time

import random
import numpy as np
import torch
from torch import nn
import pandas as pd
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

In [58]:
# import df

from pathlib import Path

data_path = Path(
    "../data/processed/household_power_features.parquet"
)

df = pd.read_parquet(data_path)
df = df.sort_values("datetime").reset_index(drop=True)

n = len(df)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

In [59]:
# import previous used features

feature_cols = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3",

    "power_lag_1",
    "power_lag_5",
    "power_lag_15",
    "power_lag_30",
    "power_lag_60",
    "power_lag_120",
    "power_lag_1380",

    "hour",
    "day_of_week",
    "month",
    "is_weekend",

    "rolling_mean_15",
    "rolling_mean_60",
    "rolling_std_60",
    "rolling_mean_1440"
]

target_col = "power_target_60"

In [60]:
# train test split

X_train = train_df[feature_cols].copy()
y_train = train_df[target_col].copy()

X_val = val_df[feature_cols].copy()
y_val = val_df[target_col].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df[target_col].copy()

In [61]:
# scales

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [43]:
X_train_tensor = torch.tensor(
    X_train_scaled,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train.to_numpy(),
    dtype=torch.float32
).reshape(-1, 1)

X_val_tensor = torch.tensor(
    X_val_scaled,
    dtype=torch.float32
)

y_val_tensor = torch.tensor(
    y_val.to_numpy(),
    dtype=torch.float32
).reshape(-1, 1)

X_test_tensor = torch.tensor(
    X_test_scaled,
    dtype=torch.float32
)

y_test_tensor = torch.tensor(
    y_test.to_numpy(),
    dtype=torch.float32
).reshape(-1, 1)

In [62]:
# device

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

Using device: cuda


In [63]:
# reduced dropout nn

class ReducedDropoutMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.10),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.05),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

In [46]:
reduced_dropout_model = ReducedDropoutMLP(
    input_dim=X_train_tensor.shape[1]
).to(device)

In [47]:
reduced_dropout_criterion = nn.MSELoss()

In [64]:
# optimizer and scheduler

reduced_dropout_optimizer = torch.optim.Adam(
    reduced_dropout_model.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

reduced_dropout_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    reduced_dropout_optimizer,
    mode="min",
    factor=0.5,
    patience=5
)

In [65]:
# loading

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

val_dataset = TensorDataset(
    X_val_tensor,
    y_val_tensor
)

batch_size = 2048

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    pin_memory=True
)

max_epochs = 100
early_stopping_patience = 10

In [66]:
# previously used train model function

def train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    scheduler,
    device,
    max_epochs=100,
    early_stopping_patience=10,
):

    history = {
        "train_loss": [],
        "val_loss": [],
        "learning_rate": []
    }

    best_val_loss = float("inf")
    best_model_state = copy.deepcopy(model.state_dict())

    epochs_without_improvement = 0

    for epoch in range(max_epochs):

        epoch_start = time.time()

        # Training phase
        model.train()

        train_loss_sum = 0.0
        train_samples = 0

        for xb, yb in train_loader:

            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            optimizer.zero_grad()

            predictions = model(xb)

            loss = criterion(predictions, yb)

            loss.backward()

            optimizer.step()

            batch_size_actual = xb.size(0)

            train_loss_sum += loss.item() * batch_size_actual
            train_samples += batch_size_actual

        train_loss = train_loss_sum / train_samples

        # Validation phase
        model.eval()

        val_loss_sum = 0.0
        val_samples = 0

        with torch.no_grad():

            for xb, yb in val_loader:

                xb = xb.to(device, non_blocking=True)
                yb = yb.to(device, non_blocking=True)

                predictions = model(xb)

                loss = criterion(predictions, yb)

                batch_size_actual = xb.size(0)

                val_loss_sum += loss.item() * batch_size_actual
                val_samples += batch_size_actual

        val_loss = val_loss_sum / val_samples

        # Scheduler
        scheduler.step(val_loss)

        current_lr = optimizer.param_groups[0]["lr"]

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["learning_rate"].append(current_lr)

        # Early stopping
        if val_loss < best_val_loss:

            best_val_loss = val_loss
            best_model_state = copy.deepcopy(
                model.state_dict()
            )

            epochs_without_improvement = 0

        else:

            epochs_without_improvement += 1

        elapsed = time.time() - epoch_start

        print(
            f"Epoch {epoch + 1:3d}/{max_epochs} | "
            f"Train Loss: {train_loss:.6f} | "
            f"Val Loss: {val_loss:.6f} | "
            f"LR: {current_lr:.2e} | "
            f"Time: {elapsed:.1f}s"
        )

        if epochs_without_improvement >= early_stopping_patience:

            print("\nEarly stopping triggered.")
            break

    model.load_state_dict(best_model_state)

    print(
        f"\nBest validation loss: "
        f"{best_val_loss:.6f}"
    )

    return model, history

In [51]:
reduced_dropout_model, reduced_dropout_history = train_model(
    model=reduced_dropout_model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=reduced_dropout_criterion,
    optimizer=reduced_dropout_optimizer,
    scheduler=reduced_dropout_scheduler,
    device=device,
    max_epochs=100,
    early_stopping_patience=10
)

Epoch   1/100 | Train Loss: 0.760724 | Val Loss: 0.673380 | LR: 1.00e-03 | Time: 7.5s
Epoch   2/100 | Train Loss: 0.682613 | Val Loss: 0.646027 | LR: 1.00e-03 | Time: 7.4s
Epoch   3/100 | Train Loss: 0.662974 | Val Loss: 0.638354 | LR: 1.00e-03 | Time: 7.4s
Epoch   4/100 | Train Loss: 0.651883 | Val Loss: 0.642904 | LR: 1.00e-03 | Time: 7.4s
Epoch   5/100 | Train Loss: 0.642825 | Val Loss: 0.638348 | LR: 1.00e-03 | Time: 7.4s
Epoch   6/100 | Train Loss: 0.636199 | Val Loss: 0.643166 | LR: 1.00e-03 | Time: 7.3s
Epoch   7/100 | Train Loss: 0.629855 | Val Loss: 0.642700 | LR: 1.00e-03 | Time: 7.4s
Epoch   8/100 | Train Loss: 0.624483 | Val Loss: 0.644065 | LR: 1.00e-03 | Time: 7.6s
Epoch   9/100 | Train Loss: 0.619531 | Val Loss: 0.648915 | LR: 5.00e-04 | Time: 7.5s
Epoch  10/100 | Train Loss: 0.611844 | Val Loss: 0.646342 | LR: 5.00e-04 | Time: 7.6s
Epoch  11/100 | Train Loss: 0.608987 | Val Loss: 0.644365 | LR: 5.00e-04 | Time: 7.6s
Epoch  12/100 | Train Loss: 0.606476 | Val Loss: 0.651

In [54]:
reduced_dropout_model.eval()

rd_predictions = []
val_targets = []

with torch.no_grad():
    for xb, yb in val_loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        preds = reduced_dropout_model(xb)

        rd_predictions.append(
            preds.cpu().numpy()
        )

        val_targets.append(
            yb.cpu().numpy()
        )

rd_predictions = np.concatenate(
    rd_predictions
).ravel()

val_targets = np.concatenate(
    val_targets
).ravel()

In [55]:
rd_mae = mean_absolute_error(
    val_targets,
    rd_predictions
)

rd_rmse = mean_squared_error(
    val_targets,
    rd_predictions
) ** 0.5

rd_r2 = r2_score(
    val_targets,
    rd_predictions
)

print(f"Reduced Dropout NN MAE:  {rd_mae:.4f}")
print(f"Reduced Dropout NN RMSE: {rd_rmse:.4f}")
print(f"Reduced Dropout NN R²:   {rd_r2:.4f}")

Reduced Dropout NN MAE:  0.5234
Reduced Dropout NN RMSE: 0.7990
Reduced Dropout NN R²:   0.4191


In [56]:
baseline_nn_rmse = 0.7983
hgb_rmse = 0.788538

vs_baseline_nn = (
    (baseline_nn_rmse - 0.7990)
    / baseline_nn_rmse
) * 100

vs_hgb = (
    (hgb_rmse - rd_rmse)
    / hgb_rmse
) * 100

print(
    f"Improvement vs baseline NN: "
    f"{vs_baseline_nn:.2f}%"
)

print(
    f"Improvement vs HGB: "
    f"{vs_hgb:.2f}%"
)

Improvement vs baseline NN: -0.09%
Improvement vs HGB: -1.32%


In [57]:
# reducing dropout did not help 0.09% worse than baseline nn, 1.32 worse than HGB